In [15]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import nbformat
from nbconvert import PythonExporter

notebook_path = "/content/drive/MyDrive/Colab Notebooks/Day 1 Agentic AI.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = nbformat.read(f, as_version=4)

exporter = PythonExporter()
python_code, _ = exporter.from_notebook_node(notebook)

print("Notebook converted successfully!")

Notebook converted successfully!


In [24]:
output_path = "/content/drive/MyDrive/Colab Notebooks/day1_code.py"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(python_code)

print("Python file saved:", output_path)

Python file saved: /content/drive/MyDrive/Colab Notebooks/day1_code.py


In [26]:
import os

print(os.path.exists(output_path))

True


In [29]:
%run "/content/drive/MyDrive/Colab Notebooks/day1_code.py"

Groq API key loaded successfully.
**Vulnerability Assessment – SQL Injection**

| Item | Details |
|------|---------|
| **Vulnerability Type** | SQL Injection (Improper input handling in SQL statements) |
| **Description** | The application concatenates raw user‑supplied data directly into an SQL query without sanitizing, escaping, or using parameterized statements. This allows an attacker to inject malicious SQL code that the database will execute. |
| **Simple Explanation** | Think of the SQL query as a sentence that the database reads. The app lets anyone add extra words to that sentence without checking them first. By inserting special words (SQL commands), an attacker can change the meaning of the sentence and make the database do things it shouldn’t, like showing secret data or deleting records. |
| **Possible Security Impact** | • **Data Disclosure** – read confidential rows (e.g., user passwords, personal data). <br>• **Data Modification/Deletion** – alter or erase records. <br

In [33]:
import json
import logging
import functools
from typing import Any

from groq import RateLimitError, APIConnectionError
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print("Day 1 and Day 2 setup ready!")

Day 1 and Day 2 setup ready!


In [35]:
VULNAGENT_SYSTEM = """
ROLE:
You are VulnAgent, an AI vulnerability assessment assistant.

FACTS:
You analyze vulnerability details supplied by the user.
The user may provide vulnerability name, affected component,
severity, CVSS score, and assessment notes.

RULES:
1. Only provide defensive security guidance.
2. Do not invent CVE IDs, CVSS scores, or affected versions.
3. Treat user-provided severity and CVSS as supplied assessment data.
4. If information is missing, clearly mention it.
5. Recommend safe remediation and verification.
6. Do not claim that a system was scanned or tested.

FORMAT:
Return ONLY valid JSON.
Use the exact keys:
vulnerability_name
severity
impact
remediation
verification
confidence
"""

In [37]:
FEW_SHOT_MESSAGES = [
    HumanMessage(
        content="Analyze: Weak password policy in an application"
    ),
    AIMessage(
        content=json.dumps({
            "vulnerability_name": "Weak password policy",
            "severity": "Medium",
            "impact": "Weak passwords may allow unauthorized account access.",
            "remediation": "Enforce strong passwords and appropriate authentication controls.",
            "verification": "Review password policy and test it using an authorized account.",
            "confidence": "Medium"
        })
    ),

    HumanMessage(
        content="Analyze: Missing security headers on a website"
    ),
    AIMessage(
        content=json.dumps({
            "vulnerability_name": "Missing security headers",
            "severity": "Low",
            "impact": "Some browser-side security protections may be reduced.",
            "remediation": "Configure suitable security headers.",
            "verification": "Check the response headers after configuration.",
            "confidence": "Medium"
        })
    )
]

print("Few-shot examples added.")

Few-shot examples added.


In [39]:
vulnerabilities = [
    {
        "id": "VULN-001",
        "name": "SQL Injection",
        "severity": "High",
        "cvss": 8.5,
        "component": "Login module"
    },
    {
        "id": "VULN-002",
        "name": "Weak Password Policy",
        "severity": "Medium",
        "cvss": 5.5,
        "component": "Authentication"
    },
    {
        "id": "VULN-003",
        "name": "Missing Security Headers",
        "severity": "Low",
        "cvss": 3.1,
        "component": "Web server"
    },
    {
        "id": "VULN-004",
        "name": "Remote Code Execution",
        "severity": "Critical",
        "cvss": 9.8,
        "component": "Application service"
    }
]

print(json.dumps(vulnerabilities, indent=4))

[
    {
        "id": "VULN-001",
        "name": "SQL Injection",
        "severity": "High",
        "cvss": 8.5,
        "component": "Login module"
    },
    {
        "id": "VULN-002",
        "name": "Weak Password Policy",
        "severity": "Medium",
        "cvss": 5.5,
        "component": "Authentication"
    },
    {
        "id": "VULN-003",
        "name": "Missing Security Headers",
        "severity": "Low",
        "cvss": 3.1,
        "component": "Web server"
    },
    {
        "id": "VULN-004",
        "name": "Remote Code Execution",
        "severity": "Critical",
        "cvss": 9.8,
        "component": "Application service"
    }
]


In [41]:
def calculate_priority(
    severity: str,
    cvss: float
) -> str:
    """
    Return a priority label using the supplied severity and CVSS score.
    """

    if severity.lower() == "critical" or cvss >= 9.0:
        return "Immediate"

    elif severity.lower() == "high" or cvss >= 7.0:
        return "High"

    elif severity.lower() == "medium" or cvss >= 4.0:
        return "Moderate"

    else:
        return "Low"


print(calculate_priority("High", 8.5))

High


In [43]:
def prioritize_vulnerabilities(
    records: list[dict[str, Any]]
) -> list[dict[str, Any]]:
    """
    Add a priority label and sort vulnerability records by CVSS score.
    """

    for record in records:
        record["priority"] = calculate_priority(
            record["severity"],
            record["cvss"]
        )

    return sorted(
        records,
        key=lambda item: item.get("cvss", 0),
        reverse=True
    )


prioritized_vulnerabilities = prioritize_vulnerabilities(
    vulnerabilities
)

for item in prioritized_vulnerabilities:
    print(
        item["id"],
        item["name"],
        item["severity"],
        item["cvss"],
        item["priority"]
    )

VULN-004 Remote Code Execution Critical 9.8 Immediate
VULN-001 SQL Injection High 8.5 High
VULN-002 Weak Password Policy Medium 5.5 Moderate
VULN-003 Missing Security Headers Low 3.1 Low


In [45]:
def parse_json_response(response_text: str) -> dict:
    """
    Convert the model's JSON response into a Python dictionary.
    """

    cleaned_text = response_text.strip()

    # Remove common Markdown JSON fences
    if cleaned_text.startswith("```"):
        cleaned_text = cleaned_text.replace("```json", "")
        cleaned_text = cleaned_text.replace("```", "")
        cleaned_text = cleaned_text.strip()

    try:
        data = json.loads(cleaned_text)

        if not isinstance(data, dict):
            raise ValueError("Expected a JSON object.")

        return data

    except json.JSONDecodeError:
        print("The AI response was not valid JSON.")
        print("Raw response:", response_text)

        return {
            "error": "Invalid JSON response"
        }

In [47]:
def log_call(func):
    """
    Log function execution without displaying sensitive data.
    """

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logging.info("Starting %s", func.__name__)

        result = func(*args, **kwargs)

        logging.info("Finished %s", func.__name__)

        return result

    return wrapper


logging.basicConfig(level=logging.INFO)

print("Logging decorator created.")

Logging decorator created.


In [49]:
@log_call
def analyze_vulnerability_json(
    details: str
) -> dict:
    """
    Analyze supplied vulnerability details and return structured JSON data.
    """

    messages = [
        SystemMessage(content=VULNAGENT_SYSTEM)
    ]

    # Add few-shot examples
    messages.extend(FEW_SHOT_MESSAGES)

    # Add recent conversation memory
    messages.extend(history[1:])

    # Add current vulnerability details
    messages.append(
        HumanMessage(content=details)
    )

    try:
        response = llm.invoke(messages)

        result = parse_json_response(
            response.content
        )

        return result

    except RateLimitError:
        return {
            "error": "Groq rate limit reached. Please try again shortly."
        }

    except APIConnectionError:
        return {
            "error": "Unable to connect to the Groq API."
        }

    except Exception as error:
        logging.error(
            "Vulnerability analysis failed: %s",
            type(error).__name__
        )

        return {
            "error": "Analysis failed. Check the notebook setup and logs."
        }

In [51]:
history = []

MAX_TURNS = 4

def analyze_with_memory(
    user_text: str
) -> dict:
    """
    Analyze vulnerability details while retaining recent conversation history.
    """

    result = analyze_vulnerability_json(user_text)

    history.append(
        HumanMessage(content=user_text)
    )

    history.append(
        AIMessage(content=json.dumps(result))
    )

    # Keep only the latest conversation turns
    max_messages = MAX_TURNS * 2

    if len(history) > max_messages:
        del history[:-max_messages]

    return result


print("Conversation memory initialized.")

Conversation memory initialized.


In [53]:
test_vulnerability = """
SQL Injection vulnerability reported in the login module.

The assessment notes indicate that user input may be included
in a database query without proper parameterization.

Explain the possible impact and recommend defensive remediation.
Do not invent a CVE ID or CVSS score.
"""

result = analyze_with_memory(
    test_vulnerability
)

print(json.dumps(result, indent=4))

{
    "vulnerability_name": "SQL Injection in login module",
    "severity": "Unknown",
    "impact": "An attacker could inject malicious SQL code through the login fields, potentially bypassing authentication, retrieving or modifying user credentials, extracting sensitive data, or executing arbitrary database commands. This can lead to unauthorized account access, data breaches, privilege escalation, and full compromise of the application and its backend database.",
    "remediation": "1. Refactor all database interactions in the login module to use prepared statements or parameterized queries provided by the database driver or ORM.\n2. Validate and sanitize all user-supplied input, enforcing strict type and length checks.\n3. Apply the principle of least privilege to the database account used by the application\u2014grant only the permissions required for authentication queries.\n4. Implement account lockout or throttling mechanisms to mitigate automated injection attempts.\n5. Use a

In [57]:
for vulnerability in prioritized_vulnerabilities:

    details = f"""
    Vulnerability: {vulnerability.get("name")}
    Component: {vulnerability.get("component")}
    Supplied severity: {vulnerability.get("severity")}
    Supplied CVSS: {vulnerability.get("cvss")}

    Explain the impact and recommend defensive remediation.
    Do not invent missing vulnerability information.
    """

    analysis = analyze_with_memory(details)

    print("\n" + "=" * 50)
    print("Vulnerability:", vulnerability["name"])
    print("Priority:", vulnerability["priority"])
    print(json.dumps(analysis, indent=4))


Vulnerability: Remote Code Execution
Priority: Immediate
{
    "vulnerability_name": "Remote Code Execution",
    "severity": "Critical",
    "impact": "An attacker can supply crafted input that causes the application service to execute arbitrary operating\u2011system commands with the privileges of the service account. This can lead to full system compromise, data theft or destruction, deployment of malware, lateral movement across the network, and persistence through backdoors or scheduled tasks.",
    "remediation": "1. Apply vendor or developer patches that properly validate and sanitize all inputs used in command\u2011execution paths.\n2. Refactor code to avoid constructing shell commands from user\u2011controlled data; use safe APIs or language\u2011level libraries that do not invoke a shell.\n3. Run the service under a non\u2011privileged, isolated account and enforce the principle of least privilege.\n4. Deploy containerization, sandboxing, or OS\u2011level isolation (e.g., SE

In [58]:
print("VulnAgent is ready!")
print("Enter vulnerability details.")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("Vulnerability: ").strip()

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("VulnAgent stopped.")
        break

    if not user_input:
        print("Please enter vulnerability details.")
        continue

    response = analyze_with_memory(user_input)

    print("\nVulnAgent:")
    print(json.dumps(response, indent=4))
    print()

VulnAgent is ready!
Enter vulnerability details.
Type 'exit' to stop.

Vulnerability: A web application login page is vulnerable to SQL injection. The application directly inserts user input into SQL queries. An attacker may bypass authentication and access sensitive database information.

VulnAgent:
{
    "vulnerability_name": "SQL Injection",
    "severity": "Not provided",
    "impact": "An attacker can inject crafted SQL statements into the login form, allowing authentication bypass, retrieval or modification of sensitive data, and potentially full database compromise. This can lead to credential theft, data exfiltration, data loss, and further lateral movement within the environment.",
    "remediation": "1. Replace string concatenation with parameterized queries or prepared statements for all database interactions in the login module.\n2. Use a reputable ORM or database access library that abstracts raw SQL.\n3. Validate and sanitize all user\u2011supplied input (e.g., usernames,